# Face recognition model for the STM32H723 door unit (round 3)

Trains the **half-width ReLU MobileFaceNet** that fits the board (about 150 KB RAM and 330 KB flash with
TinyEngine), then tests it on your own 30 people.

**How it learns (knowledge distillation).** A large pre-trained face model (the *teacher*, InsightFace's
MobileFaceNet trained on WebFace600K) turns each face into 512 numbers. Our tiny model (the *student*) is trained
to produce the same 512 numbers for the same face, and at the same time to tell CASIA's ~10,500 people apart
(ArcFace). Round 3 keeps round 1's proven head (global average pooling, 512 numbers, compared directly with the
teacher), adds ArcFace, trains on all of CASIA, and after every epoch checks the model on 200 CASIA people it never
trains on, so a bad run shows up after the first epoch instead of at the end.

**Steps:** 1 setup · 2 your faces from Google Drive · 3 training faces (CASIA-WebFace, Kaggle) · 4 teacher targets ·
5 student model · 6 train · 7 test on your 30 people (FAR, FRR, accuracy) · 8 int8 export · 9 TinyEngine size check.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Run the cells in order.
Your 30 people are **never used for training**, only for testing, so the results are honest.

In [ ]:
# 1. Setup -------------------------------------------------------------------
QUICK_TEST = False          # True = tiny run (a few minutes) to check everything works before the real run
TRAIN_IMAGES = 600_000      # all of CASIA (~490,000)
EPOCHS = 20                 # roughly 8-10 min per epoch on a T4; resumes after a disconnect
BATCH = 256
WIDTH = 0.5                 # 0.5 = the half-width model that fits the board
EMB = 512                   # embedding size, compared directly with the teacher's 512
TEACHER_DIM = 512           # the teacher's embedding size
ARC_WEIGHT = 0.1            # weight of the ArcFace (tell-people-apart) loss next to the copy-the-teacher loss
TAG = "r3"                  # round-2 files get this in their names, so round-1 results are kept
ENROL_PER_PERSON = 5        # photos used to enrol each person; the rest are test probes

import os, sys, glob, json, zipfile, subprocess, time, math, random
import numpy as np
if QUICK_TEST:
    TRAIN_IMAGES, EPOCHS = 3000, 1

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

if IN_COLAB:
    sh("pip -q install onnxruntime-gpu && pip -q install -U kaggle")  # leave Colab's own TensorFlow/CUDA/numpy untouched
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/work"
    PROJECT = "/content/drive/MyDrive/IDS_recognition"   # results are saved here
else:
    WORK = os.environ.get("WORK", os.path.abspath("work"))
    PROJECT = os.environ.get("PROJECT", os.path.abspath("project"))
os.makedirs(WORK, exist_ok=True); os.makedirs(PROJECT, exist_ok=True)

import cv2, tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
if IN_COLAB and not tf.config.list_physical_devices("GPU"):
    raise SystemExit("No GPU! Runtime -> Change runtime type -> T4 GPU, then run again.")

# Teacher model: InsightFace buffalo_s (MobileFaceNet, WebFace600K). Non-commercial research licence.
TEACHER = f"{WORK}/w600k_mbf.onnx"
if not os.path.exists(TEACHER):
    sh(f"curl -sSL -o {WORK}/buffalo_s.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip")
    with zipfile.ZipFile(f"{WORK}/buffalo_s.zip") as z:
        name = [n for n in z.namelist() if n.endswith("w600k_mbf.onnx")][0]
        open(TEACHER, "wb").write(z.read(name))
print("teacher ready:", os.path.getsize(TEACHER) // 1024, "KB")

## 2. Your faces (test set)
Upload **`dataset\processed_for_colab.zip`** from the project folder (your 1,046 face crops plus `report.csv`)
to **`MyDrive/IDS_recognition/`** in Google Drive.

In [ ]:
OWN = f"{WORK}/own"
if not os.path.isdir(f"{OWN}/faces112"):
    zips = [f"{PROJECT}/{n}" for n in ("processed_for_colab.zip", "processed.zip") if os.path.exists(f"{PROJECT}/{n}")]
    assert zips, f"Upload processed_for_colab.zip to {PROJECT} first"
    src = zips[0]
    with zipfile.ZipFile(src) as z: z.extractall(OWN)
    # the zip may contain a top-level 'processed' folder
    if os.path.isdir(f"{OWN}/processed/faces112"):
        for n in os.listdir(f"{OWN}/processed"): os.rename(f"{OWN}/processed/{n}", f"{OWN}/{n}")

import csv
face_w = {}
if os.path.exists(f"{OWN}/report.csv"):
    for r in csv.DictReader(open(f"{OWN}/report.csv")):
        if r["status"].startswith("kept"): face_w[(r["person"], r["file"])] = int(r["face_w"])
own_people = sorted(d for d in os.listdir(f"{OWN}/faces112") if os.path.isdir(f"{OWN}/faces112/{d}"))
own = {p: sorted(glob.glob(f"{OWN}/faces112/{p}/*.png")) for p in own_people}
print(len(own_people), "people,", sum(map(len, own.values())), "face crops")

## 3. Training faces: CASIA-WebFace (aligned 112×112, from Kaggle)
Put ONE of these in **`MyDrive/IDS_recognition/`** (see the "Kaggle Token and Drive Upload Guide" doc):
- `kaggle.json`: kaggle.com/settings/api → Legacy API Credentials → **Create Legacy API Key** (downloads it), or
- `kaggle_token.txt`: kaggle.com/settings/api → API Tokens → **Generate New Token**, paste the token into a text file.

CASIA-WebFace is for non-commercial research use, which covers a final-year project.

In [ ]:
CASIA = f"{WORK}/casia"
def list_images(root):
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    return sorted(p for p in glob.glob(f"{root}/**/*", recursive=True) if p.lower().endswith(exts))

train_files = list_images(CASIA) if os.path.isdir(CASIA) else []
if not train_files:
    if IN_COLAB:
        os.makedirs("/root/.kaggle", exist_ok=True)
        if os.path.exists(f"{PROJECT}/kaggle.json"):
            sh(f"cp '{PROJECT}/kaggle.json' /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json")
        elif os.path.exists(f"{PROJECT}/kaggle_token.txt"):
            tok = open(f"{PROJECT}/kaggle_token.txt").read().strip().split("=")[-1].strip()
            os.environ["KAGGLE_API_TOKEN"] = tok
        else:
            raise FileNotFoundError(f"Put kaggle.json or kaggle_token.txt in {PROJECT}")
        dl = f"{WORK}/kaggle_dl"; os.makedirs(dl, exist_ok=True)
        sh(f"kaggle datasets download -d yakhyokhuja/webface-112x112 -p {dl} --unzip -q")
        os.rename(dl, CASIA)   # identity folders don't matter: distillation needs no labels
        train_files = list_images(CASIA)
    else:  # local smoke test: stand-in images
        train_files = list_images(os.environ["STANDIN"])
random.seed(0); random.shuffle(train_files)
train_files = train_files[:TRAIN_IMAGES]
assert train_files, "no training images found"
ids_all = sorted({os.path.basename(os.path.dirname(p)) for p in train_files})
VAL_IDS = set(ids_all[-200:]) if len(ids_all) > 400 else set()      # 200 CASIA people never trained on
val_files = [p for p in train_files if os.path.basename(os.path.dirname(p)) in VAL_IDS][:2000]
train_files = [p for p in train_files if os.path.basename(os.path.dirname(p)) not in VAL_IDS]
print(len(val_files), "held-out check faces from", len(VAL_IDS), "people")
ids = sorted({os.path.basename(os.path.dirname(p)) for p in train_files})
id_of = {n: k for k, n in enumerate(ids)}
labels = np.array([id_of[os.path.basename(os.path.dirname(p))] for p in train_files], np.int32)
N_CLASSES = len(ids)
if N_CLASSES < 100:
    print(f"WARNING: only {N_CLASSES} identity folders found, so ArcFace is switched off (copy-the-teacher only)")
    ARC_WEIGHT = 0.0
print(N_CLASSES, "identities")
print(len(train_files), "training faces, e.g.", train_files[0])
im = cv2.imread(train_files[0]); print("size", im.shape)

## 4. Teacher targets
The teacher embeds every training face once (plus its mirror image), so training does not have to run the teacher.

In [ ]:
import onnxruntime as ort
so = ort.SessionOptions(); so.log_severity_level = 3  # hide a harmless output-shape warning
sess = ort.InferenceSession(TEACHER, so, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
t_in = sess.get_inputs()[0].name
print("teacher runs on:", sess.get_providers()[0])

def load112(path):
    im = cv2.imread(path)
    if im.shape[:2] != (112, 112): im = cv2.resize(im, (112, 112), interpolation=cv2.INTER_AREA)
    return im

def teacher_embed(bgr_batch):
    x = np.stack([cv2.cvtColor(b, cv2.COLOR_BGR2RGB) for b in bgr_batch]).astype(np.float32)
    x = ((x - 127.5) / 127.5).transpose(0, 3, 1, 2)
    e = sess.run(None, {t_in: x})[0]
    return e / np.linalg.norm(e, axis=1, keepdims=True)

T_PATH = f"{PROJECT}/teacher_{len(train_files)}.npy"   # kept in Drive, so a crash does not redo this step
if not os.path.exists(T_PATH):
    T = np.zeros((len(train_files), 2, TEACHER_DIM), np.float16)
    t0 = time.time()
    for i in range(0, len(train_files), 256):
        ims = [load112(p) for p in train_files[i:i + 256]]
        T[i:i + len(ims), 0] = teacher_embed(ims)
        T[i:i + len(ims), 1] = teacher_embed([im[:, ::-1] for im in ims])
        if i % 25600 == 0: print(f"{i}/{len(train_files)}  {time.time()-t0:.0f}s")
    np.save(T_PATH, T)
T = np.load(T_PATH)
print("teacher targets", T.shape)

## 5. Student: half-width ReLU MobileFaceNet (TinyEngine-compatible)
PReLU → ReLU, GDConv → global average pooling, 512-d output (round 1's head, verified with TinyEngine).
The 512 numbers are compared directly with the teacher's (no extra layer in between). The ArcFace layer is
training-only and is dropped before export.

In [ ]:
from tensorflow.keras import layers, Model
BN_EPS = 1e-5
def conv_bn(x, f, k, s, act=True, name=None):
    x = layers.Conv2D(f, k, s, padding="same", use_bias=False, name=f"{name}_conv")(x)
    x = layers.BatchNormalization(epsilon=BN_EPS, name=f"{name}_bn")(x)
    return layers.ReLU(name=f"{name}_relu")(x) if act else x
def dw_bn(x, k, s, act=True, name=None):
    x = layers.DepthwiseConv2D(k, s, padding="same", use_bias=False, name=f"{name}_dw")(x)
    x = layers.BatchNormalization(epsilon=BN_EPS, name=f"{name}_bn")(x)
    return layers.ReLU(name=f"{name}_relu")(x) if act else x
def bottleneck(x, out_c, t, s, name):
    in_c = x.shape[-1]
    y = conv_bn(x, in_c * t, 1, 1, name=f"{name}_expand")
    y = dw_bn(y, 3, s, name=f"{name}_dwise")
    y = conv_bn(y, out_c, 1, 1, act=False, name=f"{name}_project")
    return layers.Add(name=f"{name}_add")([x, y]) if (s == 1 and in_c == out_c) else y
def mobilefacenet(width=0.5, emb=512, batch=None):
    c = lambda n: max(8, int(round(n * width / 8)) * 8)
    inp = layers.Input((112, 112, 3), batch_size=batch, name="face")
    x = conv_bn(inp, c(64), 3, 2, name="conv1"); x = dw_bn(x, 3, 1, name="dwconv1")
    i = 0
    for t, ch, n, s in [(2, 64, 5, 2), (4, 128, 1, 2), (2, 128, 6, 1), (4, 128, 1, 2), (2, 128, 2, 1)]:
        for j in range(n):
            x = bottleneck(x, c(ch), t, s if j == 0 else 1, name=f"bneck{i}"); i += 1
    x = conv_bn(x, c(512), 1, 1, name="conv_last")
    x = layers.GlobalAveragePooling2D(keepdims=True, name="gap")(x)
    x = layers.Conv2D(emb, 1, use_bias=False, name="emb_conv")(x)
    x = layers.BatchNormalization(epsilon=BN_EPS, name="emb_bn")(x)
    x = layers.Reshape((emb,), name="embedding")(x)
    return Model(inp, x, name=f"mfn_relu_w{width}")

class ArcMargin(layers.Layer):
    """ArcFace: cosine to each identity's centre, with an angular margin m on the true identity."""
    def __init__(self, n, s=32.0, m=0.4, **kw):
        super().__init__(**kw); self.n, self.s, self.m = n, s, m
    def build(self, shape):
        self.W = self.add_weight(name="centres", shape=(shape[0][-1], self.n), initializer="glorot_uniform")
    def call(self, inputs):
        e, y = inputs
        cos = tf.matmul(tf.math.l2_normalize(e, 1), tf.math.l2_normalize(self.W, 0))
        cos = tf.clip_by_value(cos, -1 + 1e-7, 1 - 1e-7)
        phi = tf.cos(tf.acos(cos) + self.m)
        phi = tf.where(cos > tf.cos(math.pi - self.m), phi, cos - self.m * tf.sin(self.m) * 1.0)
        onehot = tf.one_hot(tf.reshape(y, [-1]), self.n)
        return self.s * (onehot * phi + (1 - onehot) * cos)

student = mobilefacenet(WIDTH, EMB)                                    # this is what goes on the board
img_in = layers.Input((112, 112, 3), name="img"); lab_in = layers.Input((), dtype="int32", name="label")
emb = student(img_in)
kd_out = layers.Identity(name="kd")(emb)                            # compared directly with the teacher
arc_out = ArcMargin(max(N_CLASSES, 2), name="arc")([emb, lab_in])
trainer = Model([img_in, lab_in], {"kd": kd_out, "arc": arc_out}, name="trainer")
print(student.name, f"{student.count_params():,} parameters on the board")

## 6. Train
Augmentations match the door camera: random mirror, brightness/contrast changes, and **random down-scaling**
(many of your faces are only 40–60 pixels wide, so the student must cope with blurry, upscaled faces).
Checkpoints are saved to Google Drive, so a disconnect does not lose progress.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess(img):  # uint8 RGB -> [-1, 1], same as the teacher
    return (tf.cast(img, tf.float32) - 127.5) / 127.5

def make_ds(files, targets, labels):
    idx = np.arange(len(files))
    ds = tf.data.Dataset.from_tensor_slices((files, idx, labels)).shuffle(min(len(files), 50_000), seed=1, reshuffle_each_iteration=True)
    def load(path, i, y):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.resize(img, (112, 112)); img = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)
        flip = tf.random.uniform(()) < 0.5
        img = tf.cond(flip, lambda: tf.image.flip_left_right(img), lambda: img)
        tgt = tf.gather(tf.gather(targets, i), tf.cast(flip, tf.int32))  # this face's row only (mirrored or not)
        # simulate a small, upscaled face (32-112 px) half of the time
        def shrink(im):
            s = tf.random.uniform((), 32, 112, tf.int32)
            im = tf.image.resize(tf.image.resize(tf.cast(im, tf.float32), (s, s)), (112, 112))
            return tf.cast(tf.clip_by_value(im, 0, 255), tf.uint8)
        img = tf.cond(tf.random.uniform(()) < 0.5, lambda: shrink(img), lambda: img)
        x = tf.cast(img, tf.float32)
        x = tf.image.random_brightness(x, 40.0); x = tf.image.random_contrast(x, 0.7, 1.3)
        x = (tf.clip_by_value(x, 0, 255) - 127.5) / 127.5
        return (x, y), {"kd": tf.cast(tgt, tf.float32), "arc": y}
    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH, drop_remainder=True).prefetch(AUTOTUNE)

T_tf = tf.constant(T)  # (N, 2, 512) float16
ds = make_ds(np.array(train_files), T_tf, labels)

def cosine_kd_loss(t, s):
    s = tf.math.l2_normalize(s, axis=1); t = tf.math.l2_normalize(t, axis=1)
    return tf.reduce_mean(1.0 - tf.reduce_sum(s * t, axis=1))

steps = max(1, len(train_files) // BATCH) * EPOCHS
lr = tf.keras.optimizers.schedules.CosineDecay(2e-3, steps, alpha=0.02)
def arc_loss(y, logits):
    return tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True))
trainer.compile(optimizer=tf.keras.optimizers.Adam(lr), loss={"kd": cosine_kd_loss, "arc": arc_loss},
                loss_weights={"kd": 1.0, "arc": ARC_WEIGHT})
CKPT = f"{PROJECT}/trainer_{TAG}.weights.h5"; DONE = f"{PROJECT}/epochs_done_{TAG}.txt"
start = 0
if os.path.exists(CKPT) and os.path.exists(DONE) and not QUICK_TEST:
    trainer.load_weights(CKPT); start = int(open(DONE).read())
    print(f"resuming after epoch {start} of {EPOCHS}")
def casia_check():
    """EER on held-out CASIA people + how much of the output is the same for every face (collapse)."""
    if len(val_files) < 50: return None
    def run(paths):
        out = []
        for i in range(0, len(paths), 32):
            x = np.stack([cv2.cvtColor(load112(p), cv2.COLOR_BGR2RGB) for p in paths[i:i + 32]]).astype(np.float32)
            out.append(student.predict_on_batch((x - 127.5) / 127.5))
        return np.concatenate(out)
    e = run(val_files); shared = np.linalg.norm(e.mean(0)) / np.linalg.norm(e, axis=1).mean()
    e = e / np.linalg.norm(e, axis=1, keepdims=True)
    lab = np.array([os.path.basename(os.path.dirname(p)) for p in val_files]); S = e @ e.T
    iu = np.triu_indices(len(e), 1); same = (lab[:, None] == lab[None, :])[iu]; sc = S[iu]
    gen, imp = sc[same], sc[~same]; th = np.linspace(-1, 1, 801)
    far = np.array([(imp >= t).mean() for t in th]); frr = np.array([(gen < t).mean() for t in th])
    k = np.argmin(np.abs(far - frr))
    # same collapse measure on a sample of your camera faces (no scores, so no peeking at the test)
    ec = run(sum(own.values(), [])[::5])
    shared_cam = np.linalg.norm(ec.mean(0)) / np.linalg.norm(ec, axis=1).mean()
    return 100 * (far[k] + frr[k]) / 2, shared, shared_cam

class SaveEachEpoch(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        trainer.save_weights(CKPT); open(DONE, "w").write(str(epoch + 1))
        r = casia_check()
        msg = f"  epoch {epoch + 1}: saved to Drive"
        if r:
            msg += f" | held-out CASIA EER {r[0]:.1f}% | same-for-everyone part: CASIA {r[1]:.2f}, your camera {r[2]:.2f}"
            if epoch >= 1 and max(r[1], r[2]) > 0.8:   # round 1 (worked) = 0.41, round 2 (failed) = 0.98
                msg += "  <-- WARNING: outputs collapsing, stop and tell Claude"
        print(msg)
if start < EPOCHS:
    trainer.optimizer.build(trainer.trainable_variables)
    trainer.optimizer.iterations.assign(start * (len(train_files) // BATCH))  # continue the learning-rate schedule
    hist = trainer.fit(ds, epochs=EPOCHS, initial_epoch=start, callbacks=[SaveEachEpoch()])
    h = hist.history
    print("final losses: copy-teacher (1 - cosine)", round(h["kd_loss"][-1], 4), "| ArcFace", round(h["arc_loss"][-1], 3))
else:
    print("training already finished; delete epochs_done_{TAG}.txt in Drive to train again")

## 7. Test on your 30 people
For each person the first `ENROL_PER_PERSON` photos (by capture time) form their stored template, like enrolment at
the door. Every other photo is a probe: compared with its own template it is a *genuine* attempt; compared with
everyone else's template it is an *impostor* attempt.

FAR = impostor attempts accepted ÷ all impostor attempts (Eq. 3.1). FRR = genuine attempts rejected ÷ all genuine
attempts (Eq. 3.2). The **EER** is where they are equal. Results are also split by face size.

In [ ]:
def embed_student(model, paths, bs=128):
    out = []
    for i in range(0, len(paths), bs):
        x = np.stack([cv2.cvtColor(load112(p), cv2.COLOR_BGR2RGB) for p in paths[i:i + bs]]).astype(np.float32)
        out.append(model.predict((x - 127.5) / 127.5, verbose=0))
    e = np.concatenate(out); return e / np.linalg.norm(e, axis=1, keepdims=True)

def embed_teacher(paths, bs=128):
    return np.concatenate([teacher_embed([load112(p) for p in paths[i:i + bs]]) for i in range(0, len(paths), bs)])

def evaluate(embed_fn, label):
    people = [p for p in own_people if len(own[p]) > ENROL_PER_PERSON]
    templ, probes = {}, []
    for p in people:
        e = embed_fn(own[p])
        t = e[:ENROL_PER_PERSON].mean(0); templ[p] = t / np.linalg.norm(t)
        for path, v in zip(own[p][ENROL_PER_PERSON:], e[ENROL_PER_PERSON:]):
            probes.append((p, v, face_w.get((p, os.path.basename(path)), 0)))
    names = list(templ); M = np.stack([templ[n] for n in names])
    gen, imp, gen_w = [], [], []
    for p, v, w in probes:
        s = M @ v
        for n, sc in zip(names, s):
            (gen if n == p else imp).append(sc)
            if n == p: gen_w.append(w)
    gen, imp, gen_w = np.array(gen), np.array(imp), np.array(gen_w)
    th = np.linspace(-1.0, 1.001, 2002)  # top value accepts nobody, so FAR always reaches 0
    far = np.array([(imp >= t).mean() for t in th]); frr = np.array([(gen < t).mean() for t in th])
    k = np.argmin(np.abs(far - frr)); eer, t_eer = (far[k] + frr[k]) / 2, th[k]
    k1 = np.where(far <= 0.01)[0][0]; k01 = np.where(far <= 0.001)[0][0]
    acc = ((gen >= th[k1]).sum() + (imp < th[k1]).sum()) / (len(gen) + len(imp))
    res = {"model": label, "people": len(names), "genuine": len(gen), "impostor": len(imp),
           "EER_%": round(100 * eer, 2), "thr_EER": round(float(t_eer), 3),
           "FRR_at_FAR1%_%": round(100 * frr[k1], 2), "thr_FAR1%": round(float(th[k1]), 3),
           "FRR_at_FAR0.1%_%": round(100 * frr[k01], 2), "accuracy_at_FAR1%_%": round(100 * acc, 2)}
    for lo, hi in ((0, 48), (48, 1000)):
        m = (gen_w >= lo) & (gen_w < hi)
        if m.any(): res[f"FRR@FAR1%_face{lo}-{hi}px_%"] = round(100 * (gen[m] < th[k1]).mean(), 2)
    return res, (th, far, frr)

results, curves = [], {}
for label, fn in (("teacher (full MobileFaceNet, PReLU)", embed_teacher), ("student float32", lambda p: embed_student(student, p))):
    r, c = evaluate(fn, label); results.append(r); curves[label] = c; print(json.dumps(r, indent=1))

## 8. Export int8 for the microcontroller
The board runs 8-bit integers. Calibration uses training faces plus a few of your camera's faces, so the int8
ranges match real door images. The int8 model is then tested exactly like the float one.

In [ ]:
deploy = mobilefacenet(WIDTH, EMB, batch=1)       # static batch of 1: no dynamic-shape ops for TinyEngine
deploy.set_weights(student.get_weights())
calib = random.sample(train_files, min(400, len(train_files))) + random.sample(sum(own.values(), []), min(100, sum(map(len, own.values()))))
def rep():
    for p in calib:
        x = cv2.cvtColor(load112(p), cv2.COLOR_BGR2RGB).astype(np.float32)
        yield [((x - 127.5) / 127.5)[None]]
conv = tf.lite.TFLiteConverter.from_keras_model(deploy)
conv.optimizations = [tf.lite.Optimize.DEFAULT]; conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
tfl = conv.convert()
TFL = f"{PROJECT}/mfn_relu_w{WIDTH}_{TAG}_int8.tflite"; open(TFL, "wb").write(tfl)
print("int8 model:", len(tfl) // 1024, "KB ->", TFL)

itp = tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
ii, oo = itp.get_input_details()[0], itp.get_output_details()[0]
def embed_int8(paths):
    out = []
    for p in paths:
        x = (cv2.cvtColor(load112(p), cv2.COLOR_BGR2RGB).astype(np.float32) - 127.5) / 127.5
        s, z = ii["quantization"]; q = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)[None]
        itp.set_tensor(ii["index"], q); itp.invoke()
        y = itp.get_tensor(oo["index"]).astype(np.float32); s2, z2 = oo["quantization"]
        out.append((y - z2) * s2)
    e = np.concatenate(out); return e / np.linalg.norm(e, axis=1, keepdims=True)
r, c = evaluate(embed_int8, "student int8 (on-board)"); results.append(r); curves[r["model"]] = c
print(json.dumps(r, indent=1))

import pandas as pd
df = pd.DataFrame(results); df.to_csv(f"{PROJECT}/recognition_results_{TAG}.csv", index=False)
display(df.T) if IN_COLAB else print(df.T)

import matplotlib; matplotlib.use("Agg") if not IN_COLAB else None
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for label, (th, far, frr) in curves.items():
    ax[0].plot(100 * far, 100 * (1 - frr), label=label); ax[1].plot(th, 100 * frr, label=f"FRR {label}")
    ax[1].plot(th, 100 * far, "--", label=f"FAR {label}")
ax[0].set_xscale("log"); ax[0].set_xlim(0.01, 100); ax[0].set_xlabel("FAR (%)"); ax[0].set_ylabel("Genuine accepted (%)"); ax[0].set_title("ROC on your 30 people")
ax[1].set_xlabel("Cosine-similarity threshold"); ax[1].set_ylabel("%"); ax[1].set_title("FAR and FRR vs threshold"); ax[1].set_xlim(0, 1)
for a in ax: a.grid(alpha=.3)
ax[0].legend(fontsize=8); ax[1].legend(fontsize=6)
plt.tight_layout(); plt.savefig(f"{PROJECT}/recognition_curves_{TAG}.png", dpi=150); plt.show() if IN_COLAB else None

## 9. Size on the STM32H723 (TinyEngine)
Generates the C code with MCUNetV2 patch-based inference (4×4 patches, split at layer 5) and reports RAM and flash.

In [ ]:
TE = f"{WORK}/tinyengine"
if not os.path.isdir(TE):
    sh(f"git clone -q --depth 1 https://github.com/mit-han-lab/tinyengine.git {TE}")
    cg = f"{TE}/code_generator/CodeGenerator.py"; s = open(cg).read()   # NumPy 2 fixes (same as tinyengine_numpy2.patch)
    s = s.replace("tmpW += weight[j * channel + i]", "tmpW += int(weight[j * channel + i])")
    s = s.replace("self.int32_clip(bias[i] + tmpW * input_offset)", "self.int32_clip(int(bias[i]) + tmpW * int(input_offset))")
    s = s.replace("str(bias[i] + tmpW * input_offset - self.int32_clip(bias[i] + tmpW * input_offset))",
                  "str(int(bias[i]) + tmpW * int(input_offset) - self.int32_clip(int(bias[i]) + tmpW * int(input_offset)))")
    s = s.replace("return a.astype(int)", "return int(a)"); open(cg, "w").write(s)
sys.path.insert(0, TE)
from code_generator.TfliteConvertor import TfliteConvertor
from code_generator.GeneralMemoryScheduler import GeneralMemoryScheduler
from code_generator.CodeGenerator import CodeGenerator
from code_generator.InputResizer import PatchResizer
from code_generator.PatchBasedUtil import getPatchParams
gen_dir = f"{WORK}/gen"; os.makedirs(gen_dir, exist_ok=True); cwd = os.getcwd(); os.chdir(gen_dir)
tc = TfliteConvertor(TFL); tc.parseOperatorInfo(); L = tc.layer
macs = sum(l.get_macs() for l in L)
pp = getPatchParams(L, 5, 4); PatchResizer(L).patchResize(pp["layer_cnt"], pp["grain_rf"], pp["grain_rf_height"])
ms = GeneralMemoryScheduler(L, False, False, outputTables=[], inplace=True, mem_visual_path=f"{gen_dir}/mem.png", VisaulizeTrainable=False)
ms.USE_INPLACE = True; ms.allocateMemory()
CodeGenerator(memsche=ms, inplace=True, unsigned_input=False, patch_params=pp, FP_output=False, profile_mode=False,
              fp_requantize=False, tflite_op=False, dummy_address=False, outputTables=[]).codeGeneration()
os.chdir(cwd)
print(f"MACs {macs/1e6:.1f} M | peak RAM (TinyEngine arena) {ms.buffers['input_output']/1024:.1f} KB | int8 file {len(tfl)//1024} KB")
sh(f"cd {gen_dir} && zip -q -r {PROJECT}/tinyengine_codegen_w{WIDTH}_{TAG}.zip codegen")
print("Generated C code saved to", f"{PROJECT}/tinyengine_codegen_w{WIDTH}_{TAG}.zip")